# Exploracion del catalogo `priors.csv`

Notebook exploratorio sobre los priors guardados hasta ahora (`priors_tic_v2`, `priors_tic_optimizado_2_` y las corridas mas recientes con `priors_tic_v3`).

Objetivo: revisar consistencia entre sectores del mismo TIC, distribuciones de periodo/duracion/profundidad, la relacion entre periodo orbital y rotacional, y que tan completo esta el diagnostico de calidad (columnas nuevas de v3) por fila.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("priors.csv")
df.head()

## 1) Resumen general

Cuantas corridas hay, cuantos sistemas distintos, y que fraccion de las filas tiene el diagnostico completo de `priors_tic_v3` (columnas `Mask_fraction`, `Fallback_used`, `Duration_grid_tries`, `Duration_hit_ceiling`) vs. filas viejas de v2/optimizado que no las tienen.

In [ ]:
print(f"{len(df)} corridas guardadas, {df['TIC'].nunique()} sistemas (TIC) distintos, {df['Sector'].nunique()} sectores distintos")

v3_cols = ["Mask_fraction", "Width_factor_used", "Fallback_used",
           "Duration_grid_tries", "Duration_hit_ceiling"]
n_v3 = df[v3_cols].notna().any(axis=1).sum()
print(f"{n_v3} de {len(df)} filas tienen diagnostico completo del pipeline v3 "
      f"(las demas vienen de v2/optimizado, sin esas columnas)")

df["Quality"].value_counts()

## 2) Consistencia entre sectores del mismo TIC

Cuando el mismo sistema fue observado en varios sectores, `P_orb` deberia salir practicamente igual en todos. Si no, es señal de un ajuste inestable (grid de duracion muy angosto, ambiguedad P vs 2P, etc. — justo lo que vimos con `priors_tic_v2`).

`cv_pct` es el coeficiente de variacion (desviacion estandar / media, en %) de `P_orb` entre sectores de un mismo TIC — entre mas alto, menos consistente.

In [ ]:
multi = df.groupby("TIC").filter(lambda g: g["Sector"].nunique() > 1)

consist = multi.groupby("TIC")["P_orb"].agg(["mean", "std", "count"])
consist["cv_pct"] = 100 * consist["std"] / consist["mean"]
consist.sort_values("cv_pct", ascending=False)

Los primeros de la lista (`cv_pct` de 40-77%) son los sistemas donde el periodo cambia fuerte segun el sector — vale la pena revisarlos a mano o re-correrlos con `priors_tic_v3` (grid de duracion adaptativo) para ver si el problema es el mismo que diagnosticamos en TIC 50897998: grid de duracion topado en el techo.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for tic, g in multi.groupby("TIC"):
    g = g.sort_values("Sector")
    ax.plot(g["Sector"].astype(str), g["P_orb"], "o-", alpha=0.7, label=str(tic))
ax.set_xlabel("Sector")
ax.set_ylabel("P_orb (d)")
ax.set_title("P_orb por sector, mismo TIC\n(deberia verse plano si el ajuste es consistente)")
ax.legend(fontsize=7, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3) Distribuciones de periodo

`P_orb`, `P_rot`, y la razon `P_rot / P_orb` (util para detectar binarias sincronizadas, donde la rotacion queda atrapada por marea al periodo orbital).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["P_orb"].dropna(), bins=20, color="#7b3294")
axes[0].set_xlabel("P_orb (d)"); axes[0].set_title("Periodo orbital")

axes[1].hist(df["P_rot"].dropna(), bins=20, color="#1f77b4")
axes[1].set_xlabel("P_rot (d)"); axes[1].set_title("Periodo rotacional")

ratio = df["P_rot"] / df["P_orb"]
axes[2].hist(ratio.dropna(), bins=30, color="#2ca02c")
axes[2].set_xlabel("P_rot / P_orb"); axes[2].set_title("Razon rotacion / orbita")

plt.tight_layout()
plt.show()

ratio.describe()

## 4) Sincronizacion rotacion-orbita

Si el sistema esta sincronizado por marea, los puntos deberian caer sobre la linea `P_rot = P_orb`. La linea punteada roja marca `P_rot = P_orb / 2`, que es tipico cuando Lomb-Scargle se engancha al primer armonico en vez del periodo fundamental (algo que vimos que puede pasar en el diagnostico anterior).

Agregar barras de error y agregar una barra con la excentricidad
Agregarle el tic a cada estrella, no cambia la rotacion ni el periodo orbital en diferentes sectores. Las que cambian mucho npo se sabe si es por cuestiones de mal procesamiento o realmente hay un cambio en ola orbita.  Pero por el tiempo no deberia de haber un cambio en la rotacion y la traslacion lo que haria que los sistemas son relativamente estables. Por o que estos trabajos de caracterizacion son importantes. . 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["P_orb"], df["P_rot"], c="#7b3294", alpha=0.7)

lims = [0, max(df["P_orb"].max(), df["P_rot"].max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1, label="P_rot = P_orb (sincronizado)")
ax.plot(lims, [l / 2 for l in lims], "r--", lw=1, alpha=0.5, label="P_rot = P_orb / 2")

ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("P_orb (d)"); ax.set_ylabel("P_rot (d)")
ax.set_title("Periodo rotacional vs orbital")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5) Profundidad primaria vs secundaria

Para binarias eclipsantes, la razon entre la profundidad del eclipse primario y el secundario es un proxy directo de la razon de temperaturas/radios entre las dos estrellas.

In [ ]:
d = df.dropna(subset=["Depth_primary", "Depth_secondary"])

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(d["Depth_primary"] * 100, d["Depth_secondary"] * 100,
                 c=d["P_orb"], cmap="viridis", alpha=0.8)
lims = [0, max(d["Depth_primary"].max(), d["Depth_secondary"].max()) * 100 * 1.1]
ax.plot(lims, lims, "k--", lw=1, alpha=0.5, label="profundidad igual (EW)")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Profundidad primaria (%)")
ax.set_ylabel("Profundidad secundaria (%)")
ax.set_title(f"Profundidad primaria vs secundaria  (n={len(d)})")
ax.legend(fontsize=8)
cb = plt.colorbar(sc, ax=ax); cb.set_label("P_orb (d)")
plt.tight_layout()
plt.show()

## 6) Diagnostico de salud (solo filas corridas con `priors_tic_v3`)

Estas son las unicas filas con el diagnostico completo: cobertura real de la mascara de eclipse, si se activo el fallback de "usar la curva completa", y cuantas veces se tuvo que ensanchar el grid de duracion. Utilzalo para decidir que sistemas re-correr con v3 primero — prioriza los que salieron con `cv_pct` alto en la seccion 2.

In [ ]:
v3_rows = df[df[v3_cols].notna().any(axis=1)]
v3_rows[["TIC", "Sector"] + v3_cols]

## 7) Tabla resumen por sistema

Una fila por TIC: mejor `P_orb` disponible (de la corrida con mayor `Quality`, o la mas reciente si hay empate), numero de sectores observados, y si su `P_orb` fue consistente entre sectores.

In [ ]:
summary = (
    df.sort_values("Quality")
      .groupby("TIC")
      .agg(
          n_sectores=("Sector", "nunique"),
          P_orb_mean=("P_orb", "mean"),
          P_orb_std=("P_orb", "std"),
          mejor_calidad=("Quality", "min"),
      )
      .reset_index()
)
summary["consistente"] = (summary["P_orb_std"] / summary["P_orb_mean"] < 0.02) | summary["P_orb_std"].isna()
summary.sort_values("consistente")